In [1]:
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
openai_client = OpenAI()
import pickle

In [14]:
prompt='''Question:
I just discovered the course. Can I join now?

Context:
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Does the course certificate show the number of course hours?
A: No. The certificate does not state a total number of hours.'''

In [15]:
response = openai_client.responses.create(
    model="gpt-5.6-luna",
    input=prompt
)

In [16]:
response.output[0]

ResponseOutputMessage(id='msg_097a1a6e145e6291006aab4b2da99c87d1b0ac0dcbf9e20985', content=[ResponseOutputText(annotations=[], text='Yes, you can still join and start learning. If you want to receive a certificate, make sure to submit your project before the submission period closes. Registration isn’t required to access the course or submit homework.', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase='final_answer')

In [17]:
response.output[0].content[0].text

'Yes, you can still join and start learning. If you want to receive a certificate, make sure to submit your project before the submission period closes. Registration isn’t required to access the course or submit homework.'

In [18]:
response.output_text

'Yes, you can still join and start learning. If you want to receive a certificate, make sure to submit your project before the submission period closes. Registration isn’t required to access the course or submit homework.'

In [19]:
response.usage

ResponseUsage(input_tokens=179, input_tokens_details=InputTokensDetails(cache_write_tokens=0, cached_tokens=0), output_tokens=46, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=225)

In [20]:
input_price = 0.75 / 1_000_000
output_price = 4.50 / 1_000_000

cost = (
    response.usage.input_tokens * input_price +
    response.usage.output_tokens * output_price
)

cost

0.00034125000000000003

In [21]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""
message_history = [
    {"role": "developer", "content": INSTRUCTIONS},
    {"role": "user", "content": prompt}
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=message_history
)

In [22]:

with open("documents.pkl",'rb') as f:
    documents=pickle.load(f)

In [23]:
from minsearch import Index

index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)

In [32]:
def search(query, course="llm-zoomcamp"):
    boost_dict = {"question": 2.0, "section": 0.5}
    filter_dict = {"course": course}

    return index.search(
        query,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )
# search_results

In [25]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

In [26]:
USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

In [27]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()

In [ ]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

In [22]:
def llm(instructions, user_prompt, model="gpt-5.4-mini"):
    message_history = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.responses.create(
        model=model,
        input=message_history
    )

    return response.output_text

In [ ]:

def rag(query, model="gpt-5.6-luna"):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model=model)
    return answer

In [24]:
answer = rag("I just discovered the course. Can I join now?")
print(answer)

Yes, you can still join now. If you want a certificate, though, you need to submit your project while the course is still accepting submissions.


In [25]:
rag("How do I get a certificate?")

'You can get a certificate only if you finish the course with a live cohort.\n\nTo qualify, you need to:\n- complete the capstone project\n- finish the required peer reviews\n\nHomework is not required for the certificate.\n\nYou can prepare in self-paced mode, but the project submission and peer reviews must happen while a live cohort is accepting them.'

In [2]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [41]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = openai_client.responses.create(
    model="gpt-5.6-luna",
    input=messages,
    tools=[search_tool],
)

tool_calls = [item for item in response.output if item.type == "function_call"]
tool_calls

[ResponseFunctionToolCall(arguments='{"query":"Can I join the course if I just discovered it? Late enrollment / joining course after it started"}', call_id='call_smCvSILH5xXxR79mkeWKdlxn', name='search', type='function_call', id='fc_0372aca24b3ae4e9006aab555badc887d18955e5e2bc676070', async_=None, caller=None, namespace=None, status='completed')]

In [33]:
import json
args = json.loads(tool_calls[0].arguments)
print(args)
results = search(**args)
result_json = json.dumps(results, indent=2)

{'query': 'Can I join the course after it has already started? I just discovered the course and want to enroll late.'}


In [35]:
print(result_json)

[
  {
    "id": "74eb249bbf",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "I just discovered the course. Can I still join?",
    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\u2019re still accepting submissions."
  },
  {
    "id": "5cc511f85b",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "Does the course certificate show the number of course hours?",
    "answer": "No. The certificate does not state a total number of hours."
  },
  {
    "id": "04919992b3",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "How should I start the course and follow the weekly workflow?",
    "answer": "Start with the [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/), the [general Zoomcamp logistics docs](https://datatalks.club/docs/courses/zoomcamp-logistics/), and the [LLM Zoomc

In [68]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }


In [69]:
response.output

[ResponseReasoningItem(id='rs_0d7b6397d74a722f006aab58b211c887d1884d42fb0b6b5c81', summary=[], type='reasoning', content=[], encrypted_content='gAAAAABqq1izm2LS78e28jy01b-CxvjSv6ObJnnLNA8EcHNNJzWMqyvMeFQpzvEcmBHYXqXvi3OSj9dd5a2PQkqw7Alkj6cqrN4pF_XaazA4kqq0gvvPmdrnq9PjQdz-CbO5FGfEdr4GZlA5arkFDt27e5wCnUVC7Fr04itOTvDysFe_K_K8p8pGre4DTZHurLwQjX5OKuGF9DKQ6kwqh8t7njBzOBxHnckI97N2BPzkamb76nNYN7CCj-NXLVNRVOg0j2WkKZq5zOXEmbZanPIrCxwHEYf9J4KE7qi1PfObpk6gL7becMD8auqpx1JWFVt5_iWJjhmAzZnT25r5LEqVssnISoYcfFkM40Lh-7zfUmBrmI-EmeaoujsTmKZHagRIKTJWLzQLbfZeaR5sqhgBEuGiCV5Y3TgOzmsAM2xQ7DtjMFPXezyclAkcixzn42LfK8v7kKC5IW5GXrmCACl2hprCbljY-tvWOlk5vIpw0ndyxI7z1Qb6EDIqxDYT45RksKGJPyFz6KYMpXQITGQ4BRg_X04ZPbf5HL5GajPH7lyyMs8G0GHaBHphzMNTp1dmErDB9SzAvGRIdmMX8EaYST-RBQnUK_ZXZM7RJuucgGCe3u_0y-mRpd1bssMzcx9qBPMshJidtv6Kpged3hJn2bhasHC_2KDEa5P53g2JpQQ--9l-fgjBsatnkEcZuclobCCxKvUZqdC9-GsSVwLR34LYEE6P7cajcTBPV1LUe1Cw_hAXpdyd5f88L0Vrk0W8oQirDlNjzCAL_Ab8WtOWanLsKP35K2Ir0O5YqmxSmp2oyw75ownE6GEERE5jaOpUWV-twZcKyrWemT4KbSiw

In [70]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [60]:
question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

response = openai_client.responses.create(
    model="gpt-5.6-luna",
    input=messages,
    tools=[search_tool],
)

# messages.extend(response.output)
has_tool_calls = False

for item in response.output:
    if item.type == "function_call":
        print("tool_call:", item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)
        has_tool_calls = True

    elif item.type == "message":
        messages.append({'role':'assistant', 'content':item.content[0].text})
        # print("ASSISTANT:")
        # print(item.content[0].text)

tool_call: search {"query":"I just discovered the course can I join it late enrollment joining course"}
tool_call: search {"query":"course late enrollment registration after course started new students"}
tool_call: search {"query":"joining the course enrollment eligibility"}


In [61]:
messages

[{'role': 'developer',
  'content': "You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches.\n\nTry to expand your search by using new keywords\nbased on the results you get from the search.\n\nAt the end, ask if there are other areas that the user wants to explore."},
 {'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 {'type': 'tool_call',
  'call_id': 'call_YHTINNw0jydgHc7jh90y84ks',
  'function_call_output': '[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissio

In [76]:
question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

it = 1

while True:
    print(f"iteration #{it}...")
    has_tool_calls = False

    response = openai_client.responses.create(
        model="gpt-5.6-luna",
        input=messages,
        tools=[search_tool],
    )

    # messages.extend(response.output)

    for item in response.output:
        if item.type == "function_call":
            print("tool_call:", item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_tool_calls = True

        elif item.type == "message":
            messages.append({'role':'assistant', 'content':item.content[0].text})

    it = it + 1
    if has_tool_calls == False:
        break

iteration #1...
tool_call: search {"query":"just discovered the course can I join it enrollment late joining course"}
tool_call: search {"query":"course enrollment registration join after course started"}
tool_call: search {"query":"late enrollment course joining policy"}
iteration #2...


BadRequestError: Error code: 400 - {'error': {'message': 'No tool call found for function call output with call_id call_pHDXsqMPEbE6P32ftGEyAoWP.', 'type': 'invalid_request_error', 'param': 'input', 'code': None}}

In [82]:
question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

it = 1

while True:
    print(f"iteration #{it}...")
    has_tool_calls = False

    response = openai_client.responses.create(
        model="gpt-5.6-luna",
        input=messages,
        tools=[search_tool],
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == "function_call":
            print("tool_call:", item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_tool_calls = True

        elif item.type == "message":
            messages.append({'role':'assistant', 'content':item.content[0].text})

    it = it + 1
    if has_tool_calls == False:
        break

iteration #1...
tool_call: search {"query":"I just discovered the course Can I join it late enrollment joining course"}
tool_call: search {"query":"course enrollment registration late join new students"}
tool_call: search {"query":"joining the course after it started access enrollment deadline"}
iteration #2...


In [83]:
messages

[{'role': 'developer',
  'content': "You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches.\n\nTry to expand your search by using new keywords\nbased on the results you get from the search.\n\nAt the end, ask if there are other areas that the user wants to explore."},
 {'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseReasoningItem(id='rs_027b19240b4d707f006aab5afd84c887d1a656cd9eaecd07ae', summary=[], type='reasoning', content=[], encrypted_content='gAAAAABqq1r-TbzdEKqFkxIDKwzxGBNwVL4N1i7-WKmdIqzWaLwALDCl2VZOLf3KrV6mT-cLL0iXN3-qfsXa2kxK0I1hIG8uC-39I0LmzBl_7LksYrQpZKb_d5xKlHVVQdWwO9i7U0vMZVZghNu8iJjH80XR4G0HV3IdKS_g8w7iczUyuIZOF0eS1VF8XE7JrM0QFbso50JaXZ7_RJeVmA4D7g_LKTSUrW0aMdBQlJWwwPLBJumrl5WKtaCtOhALkudXvAzi1DBgCbjGJ

In [89]:
def agent_loop(instructions, question, model="gpt-5.6-luna",verbosity=False) -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1

    while True:
        if verbosity: print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                if verbosity: print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                messages.append({'role':'assistant', 'content':last_answer})
                print(last_answer)

        it = it + 1
        if has_function_calls == False:
            break

    return last_answer

In [91]:
agent_loop(instructions, "How do I run Olama locally?")

ASSISTANT:
Assuming you mean **Ollama**, you can run it locally as follows:

### 1. Install Ollama

Download it from [ollama.com/download](https://ollama.com/download):

- **macOS/Windows:** Download and run the installer.
- **Linux:**

```bash
curl -fsSL https://ollama.com/install.sh | sh
```

### 2. Download and run a model

For example:

```bash
ollama run llama3
```

This downloads the model the first time and opens a local chat session. You can also use another model, such as:

```bash
ollama run mistral
```

### 3. Check that the local server is working

Ollama normally serves its API at `localhost:11434`. Test it with:

```bash
curl http://localhost:11434
```

You should receive a response indicating that Ollama is running.

To view downloaded models:

```bash
ollama list
```

To download a model without starting a chat:

```bash
ollama pull llama3
```

### 4. Use Ollama from Python

Install the Python client:

```bash
pip install ollama
```

Then:

```python
import ollama

resp

'Assuming you mean **Ollama**, you can run it locally as follows:\n\n### 1. Install Ollama\n\nDownload it from [ollama.com/download](https://ollama.com/download):\n\n- **macOS/Windows:** Download and run the installer.\n- **Linux:**\n\n```bash\ncurl -fsSL https://ollama.com/install.sh | sh\n```\n\n### 2. Download and run a model\n\nFor example:\n\n```bash\nollama run llama3\n```\n\nThis downloads the model the first time and opens a local chat session. You can also use another model, such as:\n\n```bash\nollama run mistral\n```\n\n### 3. Check that the local server is working\n\nOllama normally serves its API at `localhost:11434`. Test it with:\n\n```bash\ncurl http://localhost:11434\n```\n\nYou should receive a response indicating that Ollama is running.\n\nTo view downloaded models:\n\n```bash\nollama list\n```\n\nTo download a model without starting a chat:\n\n```bash\nollama pull llama3\n```\n\n### 4. Use Ollama from Python\n\nInstall the Python client:\n\n```bash\npip install olla

In [92]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "I just discovered the course. Can I join it?",verbosity=True)

iteration #1...
function_call: search {"query":"I just discovered the course. Can I join it? course enrollment late joining registration access"}
iteration #2...
function_call: search {"query":"join LLM Zoomcamp late enrollment can start whenever want videos GitHub materials deadlines"}
function_call: search {"query":"LLM Zoomcamp certificate project submission deadline live cohort accepting submissions"}
function_call: search {"query":"LLM Zoomcamp registration required accepted without registering homework form open"}
iteration #3...
ASSISTANT:
Yes—you can join and start learning whenever you want. You don’t need to wait for a confirmation email or be on a registration list; registration is mainly used to gauge interest.

Start with the [LLM Zoomcamp materials](https://datatalks.club/docs/courses/llm-zoomcamp/) and [GitHub repository](https://github.com/DataTalksClub/llm-zoomcamp). Check the [course platform](https://courses.datatalks.club/llm-zoomcamp-2026/) for current deadlines an

'Yes—you can join and start learning whenever you want. You don’t need to wait for a confirmation email or be on a registration list; registration is mainly used to gauge interest.\n\nStart with the [LLM Zoomcamp materials](https://datatalks.club/docs/courses/llm-zoomcamp/) and [GitHub repository](https://github.com/DataTalksClub/llm-zoomcamp). Check the [course platform](https://courses.datatalks.club/llm-zoomcamp-2026/) for current deadlines and submission forms. You can submit homework while its form is open.\n\nIf you want a certificate, you must submit the capstone project and complete the required peer reviews while the current live cohort is still accepting submissions. Would you like to explore the course schedule, registration process, or certificate requirements?'

In [93]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "what's queen gambit?",verbosity=True)

iteration #1...
function_call: search {"query":"queen gambit"}
iteration #2...
function_call: search {"query":"Queen's Gambit chess opening"}
function_call: search {"query":"queen gambit course"}
function_call: search {"query":"The Queen's Gambit"}
iteration #3...
ASSISTANT:
“Queen’s Gambit” isn’t covered in the course FAQ, so I can’t answer it from the available course information. It may refer to the chess opening or the TV series.

Are there other course-related areas you’d like to explore?


'“Queen’s Gambit” isn’t covered in the course FAQ, so I can’t answer it from the available course information. It may refer to the chess opening or the TV series.\n\nAre there other course-related areas you’d like to explore?'